# einops-repeat — ex9: grayscale → RGB replicate-then-shift with debug-print pipeline

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat`. Running the final beacon cell reports progress against the `Einops: Repeat` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat"
DD_SUBTOPIC = "Einops: Repeat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.repeat — quick refresher

`repeat(tensor, pattern, **axes_lengths)` introduces new axes or stretches existing ones:
1. **New axis** — `'h w -> b h w'` with `b=4` broadcasts across a new leading dim.
2. **Stretch (nearest-neighbor)** — `'h w -> (h r) w'` with `r=2` makes each row appear twice in a contiguous block (rows 0,0,1,1,2,2,...).
3. **Tile** — `'h w -> h (r w)'` with `r=2` concatenates two full copies side-by-side (cols 0..w-1, then 0..w-1 again).

Stretch vs tile: in the composite `(a b)` the axis written **first varies slower**. `(h r)` puts source row 0 at output rows `0..r-1`; `(r h)` puts source row 0 at output rows `0, h, 2h, ...`. The new exercises lean on this distinction repeatedly.

### Exercise 9 — grayscale → RGB replicate-then-shift with debug-print pipeline

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Use repeat to replicate a single-channel image across an RGB channel axis, then add a per-channel shift via broadcasting — printing shape/dtype/stride at each step to verify the pipeline.
> Keywords: channels, debug-print, broadcast-trap, visualization
> ```

**KCs targeted:** `repeat-add-axis`, `repeat-channel-replicate`, `broadcasting-rules`

Implement `ex9_gray_to_rgb_shifted(gray, shifts)`.

Inputs:
- `gray`: `(B, 1, H, W)` float image batch (grayscale, with explicit channel-of-1).
- `shifts`: `(3,)` float tensor `[r, g, b]` — per-channel additive offsets.

Output: `(B, 3, H, W)` float — `gray` replicated across the RGB axis, with `shifts[c]` added to every pixel of channel `c`.

Constraints:
1. Use **exactly one** `einops.repeat` to go from `(B, 1, H, W)` to `(B, 3, H, W)`. The `'1'` in the input pattern is significant — you're replacing the size-1 channel axis with size 3.
2. After the repeat, **print** the result's `.shape`, `.dtype`, and `.stride()` to stdout in this exact format so the test can grep your debug output: `f"after_repeat shape={tuple(r.shape)} dtype={r.dtype} stride={r.stride()}"`.
3. Add `shifts` to the replicated tensor via plain broadcasting (`+`). Print the same triple for the shifted tensor with prefix `after_shift`.

The test cell visualizes channel-0 (red) of the first batch element as a heatmap so you can confirm the shift moved the red baseline as expected.

In [ ]:
def ex9_gray_to_rgb_shifted(gray: Tensor, shifts: Tensor) -> Tensor:
    """(B, 1, H, W) → (B, 3, H, W) with per-channel additive shift.
    Must print `after_repeat ...` and `after_shift ...` debug lines."""
    raise NotImplementedError()


def _test_ex9():
    import io, contextlib

    B, H, W = 2, 4, 5
    gray = t.linspace(0.0, 1.0, B * H * W).reshape(B, 1, H, W)
    shifts = t.tensor([0.0, 0.25, -0.5])

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        out = ex9_gray_to_rgb_shifted(gray, shifts)
    log = buf.getvalue()
    print(log, end='')  # forward to the real stdout so the student sees it too

    assert out.shape == (B, 3, H, W), f'expected ({B},3,{H},{W}), got {out.shape}'
    assert out.dtype == gray.dtype, f'dtype changed: {out.dtype}'

    # Correctness: channel c equals gray + shifts[c].
    for c in range(3):
        expected = gray[:, 0] + shifts[c]
        assert t.allclose(out[:, c], expected, atol=1e-6), f'channel {c} mismatch'

    # Debug-print contract: both prefixes must appear with shape/dtype/stride.
    assert 'after_repeat' in log, f'missing `after_repeat` print:\n{log}'
    assert 'after_shift' in log, f'missing `after_shift` print:\n{log}'
    for needle in ['shape=', 'dtype=', 'stride=']:
        assert log.count(needle) >= 2, f'missing `{needle}` in at least one debug line:\n{log}'

    # Visualize channel 0 of batch 0 — should look like `gray` itself (shifts[0]=0).
    fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))
    for c, name in enumerate(['R', 'G', 'B']):
        im = axes[c].imshow(out[0, c].numpy(), cmap='gray', vmin=-0.6, vmax=1.3)
        axes[c].set_title(f'channel {name} (shift={shifts[c].item():+.2f})')
        axes[c].set_xticks([]); axes[c].set_yticks([])
        plt.colorbar(im, ax=axes[c], fraction=0.046)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex9')
    print("ex9 ✓")

_test_ex9()

<details><summary>Solution</summary>

```python
def ex9_gray_to_rgb_shifted(gray: Tensor, shifts: Tensor) -> Tensor:
    # Step 1: replace size-1 channel axis with size 3 via repeat.
    r = repeat(gray, 'b 1 h w -> b c h w', c=3)
    print(f"after_repeat shape={tuple(r.shape)} dtype={r.dtype} stride={r.stride()}")

    # Step 2: broadcast-add the per-channel shifts. shifts is (3,); we need (1, 3, 1, 1).
    shifted = r + shifts.view(1, 3, 1, 1)
    print(f"after_shift shape={tuple(shifted.shape)} dtype={shifted.dtype} stride={shifted.stride()}")
    return shifted
```

**The size-1-axis trap.** Writing `'b c h w -> b c2 h w'` with `c=1` in the input pattern is the explicit, readable way to absorb a singleton channel axis. einops accepts the literal `1` in patterns specifically for this. Without it, you'd have to `squeeze` first, then `repeat`, then `unsqueeze` — three steps where one suffices.

**Why the debug prints matter.** A stride of `0` in the channel axis after `repeat` would tell you einops returned a broadcast view (cheap); a non-zero stride means it materialized a copy. With the PyTorch backend, `einops.repeat` typically materializes via `expand` + `.contiguous()`-when-needed — the stride print lets you confirm what happened in your environment. This is the kind of multi-step pipeline introspection a flashcard can't deliver.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()